# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Get metadata as a Python dict
metadata_dict = dataset.metadata.to_json()
print(f"{metadata_dict['name']}\n{metadata_dict['description']}")
print(f"Published on: {metadata_dict['datePublished']}")
print(f"Keywords: {', '.join(metadata_dict['keywords'])}")
print(f"License: {metadata_dict['license']}")

## 2. Data Overview
Review available record sets, fields, and their IDs. This is essential to know which pieces of data we can use and how to reference them (always using `@id`).

In [ ]:
# Explore available record sets
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    for rs in record_sets:
        print(f"Record set id: {rs['@id']}")
        print(f"  Name: {rs.get('name', '(no name)')}")
        print(f"  Description: {rs.get('description', '(no description)')}")
        # List fields
        fields = rs.get('field', [])
        if fields:
            for field in fields:
                if isinstance(field, str):
                    print(f"    Field id: {field}")
                elif isinstance(field, dict):
                    print(f"    Field id: {field.get('@id', '?')}, name: {field.get('name', '?')}")
        else:
            print("    (No fields in this record set)")

## 3. Data Extraction
Load data from relevant record sets into DataFrames for analysis. All references use `@id` strings as specified by the Croissant schema.

If the dataset provides no formal record sets, check the metadata for data files or direct tabular data.

In [ ]:
# If no record sets are present, inspect 'distribution' for tabular data files:
if not record_sets:
    distributions = metadata_dict.get('distribution', [])
    print(f"Found {len(distributions)} data file(s) in distribution.")
    for dist in distributions:
        print(f"Distribution @id: {dist['@id']}")

# For this dataset, there may not be any formal record sets; mlcroissant still can often extract data
dataframes = {}
if record_sets:
    # Main path if record sets are defined
    record_set_ids = [rs['@id'] for rs in record_sets]
    for record_set_id in record_set_ids:
        print(f"Extracting records from {record_set_id} ...")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for record_set {record_set_id}, columns: {dataframes[record_set_id].columns.tolist()}")
            print(dataframes[record_set_id].head())
        else:
            print(f"No records found for record_set {record_set_id}.")
else:
    # When no record sets, try loading all records generically (may only work for certain data files)
    try:
        # This uses the mlcroissant fallback.
        records = list(dataset.records())
        df = pd.DataFrame(records)
        dataframes['default'] = df
        print(f"Loaded DataFrame with columns: {df.columns.tolist()}")
        print(df.head())
    except Exception as ex:
        print(f"Error extracting records: {ex}")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records, normalizing a numeric field, and grouping by a category.

All fields/columns should be referenced by their `@id` as in the data and metadata.

In [ ]:
# For demonstration, we need the actual column names as loaded in section 3

if dataframes:
    # Use the first available DataFrame
    record_set_key = list(dataframes.keys())[0]
    df = dataframes[record_set_key]
    print(f"Analyzing columns: {df.columns.tolist()}")
    # Try to select a numeric field (heuristic)
    numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not numeric_candidates:
        # Try to infer numeric columns based on likely names
        likely_numeric = ['MW', 'molecular_weight', 'abundance', 'normalized_abundance', 'peptide_count', 'coverage']
        for c in df.columns:
            if any(s in c.lower() for s in likely_numeric):
                try:
                    df[c] = pd.to_numeric(df[c])
                    numeric_candidates.append(c)
                except:
                    pass
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using field '{numeric_field}' for numeric analysis.")
        # Example: filter for large values
        threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (showing up to 5):")
        print(filtered_df.head())
        # Normalizing
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records (showing top 5):")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try grouping by a categorical field (like 'description', 'accession', etc)
        group_candidates = [c for c in df.columns if df[c].dtype == object and c != numeric_field]
        group_field = group_candidates[0] if group_candidates else None
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by '{group_field}':")
            print(grouped_df.head())
    else:
        print("No suitable numeric field found for analysis.")
else:
    print("No DataFrame loaded; cannot perform EDA.")

## 5. Visualization
Visualize distributions or relationships between selected fields. We use matplotlib for plotting numeric fields (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    record_set_key = list(dataframes.keys())[0]
    df = dataframes[record_set_key]
    # Try to plot the first available numeric field
    numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not numeric_candidates:
        likely_numeric = ['MW', 'molecular_weight', 'abundance', 'normalized_abundance', 'peptide_count', 'coverage']
        for c in df.columns:
            if any(s in c.lower() for s in likely_numeric):
                try:
                    df[c] = pd.to_numeric(df[c])
                    numeric_candidates.append(c)
                except:
                    pass
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel('Frequency')
        plt.show()
        # Plot relationship with another numeric field if present
        if len(numeric_candidates) > 1:
            plt.figure(figsize=(6,5))
            sns.scatterplot(x=df[numeric_candidates[0]], y=df[numeric_candidates[1]])
            plt.xlabel(numeric_candidates[0])
            plt.ylabel(numeric_candidates[1])
            plt.title(f"{numeric_candidates[0]} vs {numeric_candidates[1]}")
            plt.show()
    else:
        print("No numeric fields found for visualization.")
else:
    print("No DataFrame to plot.")

## 6. Conclusion
In this notebook, we loaded the FAIR²-compliant dataset of human mast cell extracellular vesicle proteins, explored its metadata, and demonstrated basic data extraction and exploratory analysis using the `mlcroissant` library. You can extend this notebook to perform more advanced analysis and domain-specific visualizations as necessary.